# Memory Systems for Agents

Our agent so far forgets everything the moment `run_agent` returns. Real support agents need to remember the current conversation, recent turns under a budget, and past interactions across sessions.

Three tiers, built one on top of the next:

1. **No memory** — the failure mode (baseline)
2. **Short-term**
    - 2a. **Full buffer** — append every turn (works, but tokens grow)
    - 2b. **Sliding window** — keep only the last N turns (production fix)
3. **Long-term** — store past turns in a vector DB, retrieve by similarity across sessions

End state: a customer-support agent that recognizes Charles Martinez across two separate sessions.

In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langchain_community.vectorstores import Chroma
import pandas as pd

llm = ChatOllama(model="llama3.2:3b", temperature=0.7)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

@tool
def lookupOrder(orderNo: str) -> str:
    """Look up order status by order ID. Order IDs look like #ORDNO1, #ORDNO2, etc."""
    df = pd.read_csv("order_mock_data.csv")
    if not orderNo.startswith("#"):
        orderNo = "#" + orderNo
    row = df[df["id"] == orderNo]
    if row.empty:
        return f"No order found: {orderNo}"
    r = row.iloc[0]
    return f"OrderNo: {r['id']}, Name: {r['name']}, Status: {r['status']}, Items: {r['items']}"

@tool
def lookup_customer(name: str) -> str:
    """Look up a customer profile by full name. Returns tier and spend."""
    df = pd.read_csv("customers.csv")
    row = df[df["name"].str.lower() == name.lower().strip()]
    if row.empty:
        return f"No customer profile found for '{name}'."
    r = row.iloc[0]
    return f"CustomerID: {r['customer_id']}, Tier: {r['tier']}, TotalSpent: ${r['total_spent']}"

tools = [lookupOrder, lookup_customer]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Level 1 — No Memory (the failure)

Each call to `run_agent` starts from an empty `messages` list. Turn 2 can't see turn 1.

In [2]:
def run_agent_no_memory(query: str):
    messages = [HumanMessage(content=query)]
    for _ in range(6):
        resp = llm_with_tools.invoke(messages)
        messages.append(resp)
        if not resp.tool_calls:
            return resp.content
        for c in resp.tool_calls:
            result = tool_map[c["name"]].invoke(c["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=c["id"]))
    return "(max iters)"

print(">>> Turn 1")
print(run_agent_no_memory("Look up order #ORDNO5."))
print("\n>>> Turn 2 (follow-up)")
print(run_agent_no_memory("What was the status again?"))

>>> Turn 1
The order #ORDNO5 is for Charles Martinez, and it is currently in the shipping status. The items included in the order are a Fitness Tracker and a Desk Lamp.

>>> Turn 2 (follow-up)
It seems like the previous response was not related to the original question. Here's another attempt at providing a concise answer:

Sentence-BERT (sentence transformer) is a type of deep learning model that transforms sentences into numerical vectors, capturing their semantic meaning.


## Level 2a — Short-Term: Full Conversation Buffer

Persist `messages` across calls. Each new query sees the full history. Simple, but token count grows every turn.

In [3]:
class BufferAgent:
    def __init__(self):
        self.messages = []

    def chat(self, query: str) -> str:
        self.messages.append(HumanMessage(content=query))
        for _ in range(6):
            resp = llm_with_tools.invoke(self.messages)
            self.messages.append(resp)
            if not resp.tool_calls:
                return resp.content
            for c in resp.tool_calls:
                result = tool_map[c["name"]].invoke(c["args"])
                self.messages.append(ToolMessage(content=str(result), tool_call_id=c["id"]))
        return "(max iters)"

agent = BufferAgent()
print(">>> Turn 1"); print(agent.chat("Look up order #ORDNO5."))
print(f"\n[state: {len(agent.messages)} messages in memory]")
print("\n>>> Turn 2 (follow-up)"); print(agent.chat("What was the status again?"))
print(f"\n[state: {len(agent.messages)} messages in memory]")

>>> Turn 1
Here is the formatted response:

Order #ORDNO5:

* Name: Charles Martinez
* Status: Shipping
* Items:
	+ Fitness Tracker
	+ Desk Lamp

[state: 4 messages in memory]

>>> Turn 2 (follow-up)
The status of Order #ORDNO5 is Shipping.

[state: 8 messages in memory]


## Level 2b — Short-Term: Sliding Window

The buffer works but grows unbounded. In production you keep only the last N messages (plus any system prompt). Enough context for follow-ups, bounded cost.

In [4]:
SYS = SystemMessage(content="You are a customer-support agent. Be concise.")

class WindowAgent:
    def __init__(self, window: int = 10):
        self.messages = []
        self.window = window

    def _visible(self):
        # Always send the system prompt + last N messages (preserving tool-call pairing by never starting mid-pair)
        tail = self.messages[-self.window:]
        # Guard: if tail starts with a ToolMessage, drop it (its AI call is out of window)
        while tail and isinstance(tail[0], ToolMessage):
            tail = tail[1:]
        return [SYS] + tail

    def chat(self, query: str) -> str:
        self.messages.append(HumanMessage(content=query))
        for _ in range(6):
            resp = llm_with_tools.invoke(self._visible())
            self.messages.append(resp)
            if not resp.tool_calls:
                return resp.content
            for c in resp.tool_calls:
                result = tool_map[c["name"]].invoke(c["args"])
                self.messages.append(ToolMessage(content=str(result), tool_call_id=c["id"]))
        return "(max iters)"

w = WindowAgent(window=6)
print(">>> Turn 1"); print(w.chat("Look up order #ORDNO5."))
print("\n>>> Turn 2"); print(w.chat("What was the status again?"))
print("\n>>> Turn 3"); print(w.chat("And who is the customer on that order?"))
print(f"\n[total stored: {len(w.messages)}  visible: {len(w._visible())}]")

>>> Turn 1
Order #ORDNO5 Details:

* Order Number: #ORDNO5
* Customer Name: Charles Martinez
* Current Status: Shipping
* Items:
  + Fitness Tracker
  + Desk Lamp

>>> Turn 2
The status of the order is: Shipping

>>> Turn 3
The customer on the order is: Charles Martinez

[total stored: 12  visible: 6]


## Level 3 — Long-Term: Vector Memory Across Sessions

Short-term memory dies when the process restarts. Long-term memory persists turn summaries to a vector DB. New queries retrieve relevant past turns by semantic similarity.

We create a new Chroma collection at `./conversation_memory/` — separate from the policy vectorstore.

In [5]:
from langchain_core.documents import Document

CONV_DIR = "./conversation_memory"

def get_conv_store():
    return Chroma(
        collection_name="conversations",
        embedding_function=embeddings,
        persist_directory=CONV_DIR,
    )

def remember(user_msg: str, agent_reply: str, customer: str = "unknown"):
    store = get_conv_store()
    doc = Document(
        page_content=f"Customer: {customer}\nUser: {user_msg}\nAgent: {agent_reply}",
        metadata={"customer": customer},
    )
    store.add_documents([doc])

def recall(query: str, k: int = 3):
    store = get_conv_store()
    return store.similarity_search(query, k=k)

In [6]:
class LongTermAgent:
    """Short-term window + long-term vector recall. Both customer-scoped where possible."""
    def __init__(self, customer_name: str, window: int = 6):
        self.customer = customer_name
        self.window = window
        self.messages = []

    def _visible(self, query: str):
        # 1) retrieve long-term context for this query
        recalled = recall(query, k=3)
        context = "\n\n".join(d.page_content for d in recalled) if recalled else "(no prior context)"
        sys = SystemMessage(content=(
            f"You are a customer-support agent. Current customer: {self.customer}.\n"
            f"Relevant past interactions:\n{context}"
        ))
        # 2) short-term window
        tail = self.messages[-self.window:]
        while tail and isinstance(tail[0], ToolMessage):
            tail = tail[1:]
        return [sys] + tail

    def chat(self, query: str) -> str:
        self.messages.append(HumanMessage(content=query))
        for _ in range(6):
            resp = llm_with_tools.invoke(self._visible(query))
            self.messages.append(resp)
            if not resp.tool_calls:
                remember(query, resp.content, customer=self.customer)
                return resp.content
            for c in resp.tool_calls:
                result = tool_map[c["name"]].invoke(c["args"])
                self.messages.append(ToolMessage(content=str(result), tool_call_id=c["id"]))
        return "(max iters)"

In [7]:
# === Session 1 — Charles Martinez reports a damaged item ===
s1 = LongTermAgent(customer_name="Charles Martinez")
print(">>> Session 1 / Turn 1")
print(s1.chat("Hi, I'm Charles Martinez. Order #ORDNO5 — the Fitness Tracker arrived with a cracked screen."))

>>> Session 1 / Turn 1


C:\Users\shiva\AppData\Local\Temp\ipykernel_3452\249776985.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


Hello Charles Martinez,

Thank you for reaching out to us about the issue with your Fitness Tracker order #ORDNO5. We apologize for the inconvenience and frustration caused by the cracked screen.

We're committed to making things right, and we'd like to help you resolve this issue. We'd like to offer you a replacement Fitness Tracker with a new, uncracked screen, or a full refund for the order. Please let us know which option you prefer.

Additionally, we'd like to provide you with a prepaid return shipping label so that you can easily send back the defective item.

Please let us know how you'd like to proceed, and we'll be happy to assist you further.

Your order details:

* Order No.: #ORDNO5
* Item: Fitness Tracker
* Date of Purchase: [Insert Date]
* Shipping Address: [Insert Shipping Address]

We appreciate your patience and understanding in this matter, and we look forward to hearing back from you soon.

Best regards,
[Your Name]
Customer Support Team


In [8]:
# === Session 2 — process restarts (simulated by new agent instance). Same customer returns. ===
s2 = LongTermAgent(customer_name="Charles Martinez")
print(">>> Session 2 / Turn 1 (new instance, no short-term memory of session 1)")
print(s2.chat("Can you help me like last time?"))

>>> Session 2 / Turn 1 (new instance, no short-term memory of session 1)
Hello Charles Martinez,

I see that you're experiencing an issue with your Fitness Tracker order #ORDNO5. You mentioned that the device arrived with a cracked screen. I apologize for the inconvenience this has caused.

To help resolve the issue, I'd like to offer you a replacement Fitness Tracker with a new, uncracked screen, or a full refund for the order. Please let me know which option you prefer.

Additionally, I can provide you with a prepaid return shipping label so that you can easily send back the defective item.

Your order details are as follows:

* Order No.: #ORDNO5
* Item: Fitness Tracker
* Date of Purchase: [Insert Date]
* Shipping Address: [Insert Shipping Address]

Please let me know how you'd like to proceed, and I'll be happy to assist you further.

Is there anything else I can help you with, Charles?


In [9]:
# Inspect what the agent retrieved
print("Retrieved past interactions for 'help me like last time':\n")
for i, d in enumerate(recall("help me like last time", k=3), 1):
    print(f"--- Match {i} ---")
    print(d.page_content)
    print()

Retrieved past interactions for 'help me like last time':

--- Match 1 ---
Customer: Charles Martinez
User: Can you help me like last time?
Agent: Hello Charles Martinez,

I see that you're experiencing an issue with your Fitness Tracker order #ORDNO5. You mentioned that the device arrived with a cracked screen. I apologize for the inconvenience this has caused.

To help resolve the issue, I'd like to offer you a replacement Fitness Tracker with a new, uncracked screen, or a full refund for the order. Please let me know which option you prefer.

Additionally, I can provide you with a prepaid return shipping label so that you can easily send back the defective item.

Your order details are as follows:

* Order No.: #ORDNO5
* Item: Fitness Tracker
* Date of Purchase: [Insert Date]
* Shipping Address: [Insert Shipping Address]

Please let me know how you'd like to proceed, and I'll be happy to assist you further.

Is there anything else I can help you with, Charles?

--- Match 2 ---
Custo

## Summary

| Level | Where it lives | Lifespan | Cost | When to use |
|---|---|---|---|---|
| None | — | — | free | stateless Q&A, no follow-ups |
| Short-term buffer | `messages` list | session | tokens grow | short conversations |
| Sliding window | last N messages | session | bounded | long sessions, cost-conscious |
| Long-term vector | Chroma on disk | persistent | embedding cost | cross-session personalization |

**Production agents stack all three:** system prompt + sliding window of current session + retrieved long-term memories relevant to the current query.

That's the full agent picture — Model, Tools, Reasoning Loop, and Memory — all built from scratch across Notebooks 1–6.